# Telco Customer Churn Prediction (Improved)

**What changed vs. the original notebook**

| Issue in original | Fix here |
|---|---|
| Best model picked using **test-set** F1 (leaks test info into selection) | Models compared with **5-fold CV on the training set only**; test set is touched once at the end |
| Tuned for F1 at a fixed 0.5 cut-off (tuned model scored *worse* on test) | Tune on **PR-AUC**, then pick the **decision threshold** from out-of-fold predictions |
| 11 rows dropped (blank `TotalCharges`) | Those are `tenure = 0` new customers, so `TotalCharges = 0` and all 7,043 rows are kept |
| No baseline, no feature engineering | Logistic Regression baseline + a few business-driven features |
| Duplicate / empty cells, hard-coded fragile download URL | Clean structure, robust data loader with upload fallback |
| Only accuracy / F1 at 0.5 | PR-AUC, calibration, lift table, cost-based threshold option, permutation importance |
| Saved model only | Saved bundle = model + threshold + feature list, plus a `predict_churn()` helper |

## 1. Setup

In [3]:
import os
import warnings
import urllib.request

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from IPython.display import display
from scipy.stats import randint, uniform, loguniform

from sklearn.base import clone
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, cross_validate,
    cross_val_predict, RandomizedSearchCV,
)
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    RandomForestClassifier, AdaBoostClassifier,
    GradientBoostingClassifier, HistGradientBoostingClassifier,
)
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix,
    classification_report, ConfusionMatrixDisplay,
    RocCurveDisplay, PrecisionRecallDisplay,
)
from sklearn.calibration import CalibrationDisplay
from sklearn.inspection import permutation_importance
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

SEED = 42
np.random.seed(SEED)

# Business assumptions for the (optional) cost-based threshold.
# FN = churner we failed to flag (lost customer), FP = retention offer wasted.
COST_FN = 5.0
COST_FP = 1.0
THRESHOLD_STRATEGY = "f1"   # "f1" or "cost"

print("Setup complete.")

ModuleNotFoundError: No module named 'xgboost'

## 2. Load data
Tries a local file, then public mirrors. If both fail (e.g. no internet), Colab will prompt you to upload the CSV.

In [ ]:
FILE_PATH = "WA_Fn-UseC_-Telco-Customer-Churn.csv"
URLS = [
    "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv",
]

if not os.path.exists(FILE_PATH):
    for url in URLS:
        try:
            print("Downloading from", url)
            urllib.request.urlretrieve(url, FILE_PATH)
            break
        except Exception as e:
            print("  failed:", e)

if not os.path.exists(FILE_PATH):
    try:
        from google.colab import files
        print("Please upload the Telco churn CSV...")
        uploaded = files.upload()
        FILE_PATH = list(uploaded.keys())[0]
    except ImportError:
        raise FileNotFoundError("Place the Telco churn CSV next to the notebook as " + FILE_PATH)

df_raw = pd.read_csv(FILE_PATH)
print("Shape:", df_raw.shape)
display(df_raw.head())

NameError: name 'os' is not defined

## 3. Cleaning

In [ ]:
df = df_raw.copy()
df = df.drop(columns=["customerID"])

# TotalCharges has blank strings; they correspond to brand-new customers (tenure == 0).
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"].astype(str).str.strip(), errors="coerce")
blank = df["TotalCharges"].isna()
print("Blank TotalCharges rows:", blank.sum(), "| all have tenure == 0:", bool((df.loc[blank, "tenure"] == 0).all()))
df.loc[blank, "TotalCharges"] = 0.0

df["Churn"] = df["Churn"].map({"No": 0, "Yes": 1})

print("Missing values:", int(df.isna().sum().sum()))
print("Duplicate rows:", int(df.duplicated().sum()))
print(f"Churn rate: {df['Churn'].mean():.2%}")

## 4. Exploratory analysis

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for ax, col in zip(axes.ravel(), ["Contract", "InternetService", "PaymentMethod",
                                  "OnlineSecurity", "TechSupport", "PaperlessBilling"]):
    rate = df.groupby(col)["Churn"].mean().sort_values(ascending=False) * 100
    sns.barplot(x=rate.values, y=rate.index, ax=ax, color="#4C72B0")
    ax.axvline(df["Churn"].mean() * 100, ls="--", c="k", lw=1)
    ax.set_title(f"Churn rate by {col}")
    ax.set_xlabel("Churn rate (%)")
    ax.set_ylabel("")
plt.tight_layout()
plt.show()

In [ ]:
plot_df = df.assign(Status=df["Churn"].map({0: "Stayed", 1: "Churned"}))
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, col in zip(axes, ["tenure", "MonthlyCharges", "TotalCharges"]):
    sns.kdeplot(data=plot_df, x=col, hue="Status", common_norm=False, fill=True, ax=ax)
    ax.set_title(f"{col} by churn status")
plt.tight_layout()
plt.show()

## 5. Feature engineering
Row-wise features only (no statistics learned from data), so they are safe to compute before the split.

In [ ]:
ADDON_COLS = ["OnlineSecurity", "OnlineBackup", "DeviceProtection",
              "TechSupport", "StreamingTV", "StreamingMovies"]

def add_features(data):
    d = data.copy()
    d["n_addons"] = (d[ADDON_COLS] == "Yes").sum(axis=1)
    d["has_internet"] = (d["InternetService"] != "No").astype(int)
    d["avg_monthly_spend"] = (d["TotalCharges"] / d["tenure"].replace(0, np.nan)).fillna(d["MonthlyCharges"])
    d["charge_ratio"] = d["MonthlyCharges"] / d["avg_monthly_spend"]
    d["is_autopay"] = d["PaymentMethod"].str.contains("automatic", case=False).astype(int)
    d["tenure_group"] = pd.cut(
        d["tenure"], [-1, 6, 12, 24, 48, 72],
        labels=["0-6", "7-12", "13-24", "25-48", "49-72"],
    ).astype(str)
    return d

X = add_features(df.drop(columns=["Churn"]))
y = df["Churn"]

numeric_features = X.select_dtypes(include="number").columns.tolist()
categorical_features = X.select_dtypes(exclude="number").columns.tolist()
print("Numeric:", numeric_features)
print("Categorical:", categorical_features)

## 6. Train / test split
The test set is held out and **not used for any modelling decision** until the final evaluation.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=SEED, stratify=y
)
print("Train:", X_train.shape, "| Test:", X_test.shape)
print("Churn rate  train: %.3f  test: %.3f" % (y_train.mean(), y_test.mean()))

## 7. Candidate models (with baseline)

In [ ]:
preprocessor = ColumnTransformer([
    ("num", StandardScaler(), numeric_features),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
])

def make(estimator):
    return Pipeline([("prep", clone(preprocessor)), ("model", estimator)])

pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

models = {
    "Logistic Regression": make(LogisticRegression(max_iter=2000, class_weight="balanced")),
    "Random Forest": make(RandomForestClassifier(
        n_estimators=300, min_samples_leaf=5, class_weight="balanced_subsample",
        random_state=SEED, n_jobs=1)),
    "AdaBoost": make(AdaBoostClassifier(n_estimators=150, learning_rate=0.5, random_state=SEED)),
    "Gradient Boosting": make(GradientBoostingClassifier(
        n_estimators=150, learning_rate=0.1, max_depth=3, random_state=SEED)),
    "HistGradientBoosting": make(HistGradientBoostingClassifier(
        learning_rate=0.05, max_iter=250, class_weight="balanced", random_state=SEED)),
    "XGBoost": make(XGBClassifier(
        n_estimators=300, max_depth=3, learning_rate=0.05, subsample=0.8,
        colsample_bytree=0.8, scale_pos_weight=pos_weight, eval_metric="logloss",
        random_state=SEED, n_jobs=1)),
}
print("Models:", list(models))

## 8. Model comparison — 5-fold stratified CV on the training set

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
scoring = {"roc_auc": "roc_auc", "pr_auc": "average_precision",
           "f1": "f1", "precision": "precision", "recall": "recall"}

rows = []
for name, model in models.items():
    s = cross_validate(model, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)
    rows.append({"Model": name,
                 **{k: s[f"test_{k}"].mean() for k in scoring},
                 "pr_auc_std": s["test_pr_auc"].std()})
    print(f"done: {name}")

cv_df = pd.DataFrame(rows).sort_values("pr_auc", ascending=False).reset_index(drop=True)
display(cv_df.round(4))

cv_df.set_index("Model")[["roc_auc", "pr_auc", "f1", "precision", "recall"]].plot(
    kind="bar", figsize=(12, 5))
plt.title("Cross-validated performance (training set only; F1/precision/recall at 0.5)")
plt.ylim(0, 1); plt.xticks(rotation=20); plt.tight_layout(); plt.show()

## 9. Hyper-parameter tuning
We tune the two best models on **PR-AUC** (threshold-independent and appropriate for the 27% positive class),
rather than F1 at a fixed 0.5 cut-off.

In [ ]:
spaces = {
    "Logistic Regression": {"model__C": loguniform(1e-2, 1e2)},
    "Random Forest": {
        "model__n_estimators": randint(200, 600),
        "model__max_depth": [None, 6, 10, 14],
        "model__min_samples_leaf": randint(2, 20),
        "model__max_features": ["sqrt", 0.3, 0.5],
    },
    "AdaBoost": {
        "model__n_estimators": randint(50, 400),
        "model__learning_rate": loguniform(0.02, 1.0),
    },
    "Gradient Boosting": {
        "model__n_estimators": randint(100, 500),
        "model__learning_rate": loguniform(0.01, 0.2),
        "model__max_depth": randint(2, 5),
        "model__min_samples_leaf": randint(5, 40),
        "model__subsample": uniform(0.6, 0.4),
    },
    "HistGradientBoosting": {
        "model__learning_rate": loguniform(0.01, 0.2),
        "model__max_iter": randint(100, 500),
        "model__max_depth": [None, 3, 4, 6],
        "model__min_samples_leaf": randint(10, 60),
        "model__l2_regularization": loguniform(1e-3, 10),
    },
    "XGBoost": {
        "model__n_estimators": randint(150, 600),
        "model__learning_rate": loguniform(0.01, 0.2),
        "model__max_depth": randint(2, 6),
        "model__min_child_weight": randint(1, 10),
        "model__subsample": uniform(0.6, 0.4),
        "model__colsample_bytree": uniform(0.5, 0.5),
        "model__reg_lambda": loguniform(0.1, 20),
    },
}

candidates = {}   # name -> (fitted estimator, CV PR-AUC)
for _, r in cv_df.iterrows():
    candidates[r["Model"]] = (models[r["Model"]], r["pr_auc"])

for name in cv_df["Model"].head(2):
    search = RandomizedSearchCV(
        models[name], spaces[name], n_iter=20, scoring="average_precision",
        cv=cv, random_state=SEED, n_jobs=-1, refit=True)
    search.fit(X_train, y_train)
    print(f"{name}: baseline CV PR-AUC {candidates[name][1]:.4f} -> tuned {search.best_score_:.4f}")
    print("   best params:", {k.replace('model__', ''): (round(v, 4) if isinstance(v, float) else v)
                              for k, v in search.best_params_.items()})
    candidates[name + " (tuned)"] = (search.best_estimator_, search.best_score_)

final_name = max(candidates, key=lambda k: candidates[k][1])
final_model = clone(candidates[final_name][0])
print("\nSelected model (by CV PR-AUC on training data):", final_name)

## 10. Choose the decision threshold (out-of-fold, training data only)
Class-weighted / imbalanced models rarely have their best operating point at 0.5.

In [ ]:
oof_prob = cross_val_predict(final_model, X_train, y_train, cv=cv, method="predict_proba")[:, 1]

grid = np.linspace(0.05, 0.95, 91)
rows = []
for t in grid:
    pred = (oof_prob >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_train, pred, labels=[0, 1]).ravel()
    rows.append({"threshold": t,
                 "precision": precision_score(y_train, pred, zero_division=0),
                 "recall": recall_score(y_train, pred),
                 "f1": f1_score(y_train, pred),
                 "cost": fn * COST_FN + fp * COST_FP})
thr_df = pd.DataFrame(rows)

best_f1_thr = float(thr_df.loc[thr_df["f1"].idxmax(), "threshold"])
best_cost_thr = float(thr_df.loc[thr_df["cost"].idxmin(), "threshold"])
THRESHOLD = best_f1_thr if THRESHOLD_STRATEGY == "f1" else best_cost_thr
print(f"F1-optimal threshold: {best_f1_thr:.2f} | cost-optimal threshold: {best_cost_thr:.2f}")
print(f"Using '{THRESHOLD_STRATEGY}' strategy -> threshold = {THRESHOLD:.2f}")

fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
thr_df.plot(x="threshold", y=["precision", "recall", "f1"], ax=ax[0])
ax[0].axvline(THRESHOLD, ls="--", c="k"); ax[0].set_title("Precision / recall / F1 vs threshold (OOF)")
thr_df.plot(x="threshold", y="cost", ax=ax[1], color="firebrick")
ax[1].axvline(THRESHOLD, ls="--", c="k")
ax[1].set_title(f"Expected cost (FN={COST_FN:g}, FP={COST_FP:g}) vs threshold")
plt.tight_layout(); plt.show()

## 11. Final evaluation on the held-out test set (used once)

In [ ]:
final_model.fit(X_train, y_train)
test_prob = final_model.predict_proba(X_test)[:, 1]

def evaluate(y_true, prob, thr):
    pred = (prob >= thr).astype(int)
    return {"Accuracy": accuracy_score(y_true, pred),
            "Precision": precision_score(y_true, pred, zero_division=0),
            "Recall": recall_score(y_true, pred),
            "F1": f1_score(y_true, pred),
            "ROC-AUC": roc_auc_score(y_true, prob),
            "PR-AUC": average_precision_score(y_true, prob)}

summary = pd.DataFrame({"Threshold 0.50": evaluate(y_test, test_prob, 0.5),
                        f"Tuned threshold {THRESHOLD:.2f}": evaluate(y_test, test_prob, THRESHOLD)}).T
print("Final model:", final_name)
display(summary.round(4))

test_pred = (test_prob >= THRESHOLD).astype(int)
print(classification_report(y_test, test_pred, target_names=["No Churn", "Churn"]))

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(22, 4.8))
ConfusionMatrixDisplay.from_predictions(
    y_test, (test_prob >= 0.5).astype(int), display_labels=["No Churn", "Churn"],
    cmap="Blues", ax=axes[0], colorbar=False)
axes[0].set_title("Confusion matrix @ 0.50")
ConfusionMatrixDisplay.from_predictions(
    y_test, test_pred, display_labels=["No Churn", "Churn"],
    cmap="Blues", ax=axes[1], colorbar=False)
axes[1].set_title(f"Confusion matrix @ {THRESHOLD:.2f}")
RocCurveDisplay.from_predictions(y_test, test_prob, ax=axes[2], name=final_name)
axes[2].plot([0, 1], [0, 1], "k--", lw=1); axes[2].set_title("ROC curve")
PrecisionRecallDisplay.from_predictions(y_test, test_prob, ax=axes[3], name=final_name)
axes[3].axhline(y_test.mean(), ls="--", c="k", lw=1); axes[3].set_title("Precision-recall curve")
plt.tight_layout(); plt.show()

### Calibration
Class-weighted models over-predict probabilities. Use the scores for **ranking/targeting**; recalibrate
(e.g. `CalibratedClassifierCV`) if you need true probabilities.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
CalibrationDisplay.from_predictions(y_test, test_prob, n_bins=10, ax=ax, name=final_name)
ax.set_title("Calibration curve (test)")
plt.tight_layout(); plt.show()

### Business view: lift by decile
If the retention team can only contact the top-scored customers, how many churners do they reach?

In [ ]:
lift = pd.DataFrame({"y": y_test.values, "p": test_prob}).sort_values("p", ascending=False).reset_index(drop=True)
lift["decile"] = pd.qcut(lift.index, 10, labels=range(1, 11))
tbl = lift.groupby("decile", observed=True).agg(customers=("y", "size"), churners=("y", "sum"))
tbl["churn_rate"] = tbl["churners"] / tbl["customers"]
tbl["cum_capture"] = tbl["churners"].cumsum() / tbl["churners"].sum()
tbl["lift"] = tbl["churn_rate"] / y_test.mean()
display(tbl.round(3))
print(f"Top 20% of customers by score capture {tbl['cum_capture'].iloc[1]:.0%} of all churners.")

## 12. What drives churn? (permutation importance on raw features)

In [ ]:
perm = permutation_importance(final_model, X_test, y_test, scoring="average_precision",
                              n_repeats=10, random_state=SEED, n_jobs=-1)
imp = (pd.DataFrame({"Feature": X_test.columns, "Importance": perm.importances_mean,
                     "Std": perm.importances_std})
       .sort_values("Importance", ascending=False))
display(imp.head(15).round(4))

top = imp.head(15).iloc[::-1]
plt.figure(figsize=(9, 6))
plt.barh(top["Feature"], top["Importance"], xerr=top["Std"], color="#4C72B0")
plt.xlabel("Drop in PR-AUC when shuffled"); plt.title("Top 15 permutation importances (test)")
plt.tight_layout(); plt.show()

## 13. Save the model bundle and reload it

In [ ]:
bundle = {
    "model": final_model,
    "threshold": THRESHOLD,
    "model_name": final_name,
    "input_columns": df_raw.drop(columns=["customerID", "Churn"]).columns.tolist(),
    "test_metrics": summary.loc[f"Tuned threshold {THRESHOLD:.2f}"].to_dict(),
}
joblib.dump(bundle, "customer_churn_model.pkl")
print("Saved customer_churn_model.pkl")

def predict_churn(raw_customers, bundle_path="customer_churn_model.pkl"):
    """raw_customers: DataFrame with the original Telco columns (customerID optional)."""
    b = joblib.load(bundle_path)
    d = raw_customers.copy()
    d["TotalCharges"] = pd.to_numeric(d["TotalCharges"].astype(str).str.strip(), errors="coerce")
    d["TotalCharges"] = d["TotalCharges"].fillna(0.0)
    feats = add_features(d[b["input_columns"]])
    prob = b["model"].predict_proba(feats)[:, 1]
    return pd.DataFrame({"churn_probability": prob,
                         "predicted_churn": (prob >= b["threshold"]).astype(int)},
                        index=raw_customers.index)

# Demo on a few raw rows
display(predict_churn(df_raw.head(5)))

## 14. Summary & next steps
* Models are compared and tuned with cross-validation on the training set; the test set is evaluated once.
* The decision threshold is chosen from out-of-fold predictions and can be switched to a cost-based
  rule by editing `COST_FN`, `COST_FP` and `THRESHOLD_STRATEGY` in the setup cell.
* Month-to-month contracts, short tenure, fiber internet, no online security/tech support and electronic-check
  payment are the strongest churn signals — good targets for retention offers (e.g. incentives to move to annual contracts).
* Ideas to go further: SHAP explanations, probability calibration, time-based validation if dates are available,
  and A/B testing the retention campaign.